In [17]:
import torch
import torch.nn as nn
from torchvision.datasets import CIFAR100
from torch.utils.data import Dataset,DataLoader
import torch.optim as optim
from sklearn.metrics import(
    accuracy_score,
    classification_report,
    confusion_matrix
)
from torchvision.models import vgg19,VGG19_Weights
import torchvision.transforms as T

In [3]:
device = torch.device('mps' if torch.mps.is_available else 'cpu')
device

device(type='mps')

In [4]:
model = vgg19(weights=VGG19_Weights.DEFAULT)
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [10]:
for name,param in model.named_parameters():
    if not name.startswith('classifier'):
       param.requires_grad = False

In [12]:
for param in model.parameters():
    print(param.requires_grad)

False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
True
True
True
True
True
True


In [19]:
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225])]
)
train_dataset = CIFAR100(root='./data', train=True, download=True, transform=transform)
test_dataset = CIFAR100(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)
num_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(in_features=num_features,out_features=len(test_dataset.classes))

100%|███████████████████████████████████████| 169M/169M [00:49<00:00, 3.39MB/s]


In [15]:
num_features = model.classifier[6].in_features
num_features

4096

In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(),lr=1e-3)

In [25]:
model.to(device)
def train_loop(model,train_loader,criterion,optimizer):
    model.train()